In [1]:
import pandas as pd
import time
from dataLoader import load_msi_data, MSI_SATELLITE_TEST_DIR, MSI_SATELLITE_TRAIN_DIR, GT_TRAIN_CSV_PATH, column_names
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor
from joblib import parallel_backend

gt_train_df = pd.read_csv(GT_TRAIN_CSV_PATH)

X_train = load_msi_data(MSI_SATELLITE_TRAIN_DIR)
X_test = load_msi_data(MSI_SATELLITE_TEST_DIR)
y_train = gt_train_df[column_names].values



In [2]:
import numpy as np
from BaselineRegressor import BaselineRegressor

# Fit the baseline model
# baseline_reg = BaselineRegressor()
# baseline_reg = baseline_reg.fit(X_train, y_train)
# baseline_predictions = baseline_reg.predict(X_test)

# # Generate baseline values to be used in score computation
# baselines = np.mean((y_test - baseline_predictions) ** 2, axis=0)


# # Generate predictions slightly different from baseline predictions
# np.random.seed(0)
# predictions = np.zeros_like(y_test)
# for column_index in range(predictions.shape[1]):
#     class_mean_value = baseline_reg.mean[column_index]
#     predictions[:, column_index] = np.random.uniform(low=class_mean_value - class_mean_value * 0.2,
#                                                      high=class_mean_value + class_mean_value * 0.2,
#                                                      size=len(predictions))

# # Calculate MSE for each class
# mse = np.mean((y_test - predictions) ** 2, axis=0)

# # Calculate the score for each class individually
# scores = mse / baselines

# # Calculate the final score
# final_score = np.mean(scores)

# for score, class_name in zip(scores, column_names):
#     print(f"Class {class_name} score: {score}")

# print(f"Final score: {final_score}")

In [3]:
import torch
from Model import Model
from skorch import NeuralNetRegressor
from MultiOutputTuner import MultiOutputTuner

torch.manual_seed(42)

X_train = torch.FloatTensor(X_train)
y_train = torch.tensor(y_train, dtype=torch.float32)

In [ ]:
# criterion = torch.nn.MSELoss()
# optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

def toFloat32(x):
    return x.astype(np.float32)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('toFloat32', FunctionTransformer(toFloat32)),
    ('model', NeuralNetRegressor(
        Model,
        lr=0.001,
        max_epochs=50,
        verbose=0,
        device="cuda"
    ))])

param_grid = {
    'regressor__model__lr': [1e-4, 1e-5, 1e-6],
    'regressor__model__max_epochs': [100],
    'regressor__model__optimizer': [torch.optim.Adam],
    'regressor__model__module__h1': [256, 512, 1024],
    'regressor__model__module__h2': [256, 512, 1024],
    'regressor__model__module__h3': [256, 512, 1024],
}

model = TransformedTargetRegressor(
    regressor = pipeline,
    transformer = StandardScaler()
)
gs = GridSearchCV(
    model,
    param_grid,
    cv=3,
    verbose=2,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

start = time.perf_counter()
with parallel_backend("threading"):
    gs.fit(X_train, y_train)
end = time.perf_counter()

print(f"time: {end-start:4f}")
# gs.save("neural network")
print(gs.best_score_)
print(gs.best_params_)

# for i in range(epochs):
#     y_pred = model(X_train)

#     loss = criterion(y_pred, y_train)
#     losses.append(loss.item())

#     optimizer.zero_grad()
#     loss.backward()
#     optimizer.step()

#     if i % 50 == 0:
#         print(f"Epoch {i}: {loss.item()}")

# print(f"Epoch {epochs - 1}: {loss.item()}")

Fitting 3 folds for each of 81 candidates, totalling 243 fits


/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory


[CV] END regressor__model__lr=0.0001, regressor__model__max_epochs=100, regressor__model__module__h1=256, regressor__model__module__h2=512, regressor__model__module__h3=256, regressor__model__optimizer=<class 'torch.optim.adam.Adam'>; total time=  34.7s
[CV] END regressor__model__lr=0.0001, regressor__model__max_epochs=100, regressor__model__module__h1=256, regressor__model__module__h2=512, regressor__model__module__h3=256, regressor__model__optimizer=<class 'torch.optim.adam.Adam'>; total time=  34.9s
[CV] END regressor__model__lr=0.0001, regressor__model__max_epochs=100, regressor__model__module__h1=256, regressor__model__module__h2=256, regressor__model__module__h3=256, regressor__model__optimizer=<class 'torch.optim.adam.Adam'>; total time=  35.0s
[CV] END regressor__model__lr=0.0001, regressor__model__max_epochs=100, regressor__model__module__h1=256, regressor__model__module__h2=512, regressor__model__module__h3=512, regressor__model__optimizer=<class 'torch.optim.adam.Adam'>; tot

In [ ]:
# max epochs: 2000
# time: 2979.517762
# -367.6711120605469
#{'regressor__model__lr': 0.0001, 'regressor__model__max_epochs': 100, 'regressor__model__module__h1': 1024, 'regressor__model__module__h2': 1024, 'regressor__model__module__h3': 1024, 'regressor__model__optimizer': <class 'torch.optim.adam.Adam'>}